# Hosting CrewAI multi-agent crew with Amazon Bedrock models in Amazon Bedrock AgentCore Runtime

## Overview

In this tutorial we will learn how to host your existing multi-agent crew, using Amazon Bedrock AgentCore Runtime. 

We will focus on a CrewAI with Amazon Bedrock model example. For Strands Agents with Amazon Bedrock model check [here](../01-strands-with-bedrock-model) and for a Strands Agents with an OpenAI model check [here](../03-strands-with-openai-model).


### Tutorial Details

| Information | Details |
|:--------------------|:-----------------------------------------------------------------------------|
| Tutorial type | Conversational |
| Agent type | Multi-agent crew |
| Agentic Framework | CrewAI |
| LLM model | Anthropic Claude Haiku 4.5 (RP changed to Nova Lite/Pro)|
| Tutorial components | Hosting agent on AgentCore Runtime. Using CrewAI and Amazon Bedrock Model |
| Tutorial vertical | Cross-vertical |
| Example complexity | Easy |
| SDK used | Amazon BedrockAgentCore Python SDK and boto3 |

### Tutorial Architecture

In this tutorial we will describe how to deploy an existing multi-agent crew to AgentCore runtime. 

For demonstration purposes, we will use a CrewAI crew using Amazon Bedrock models

In our example we will use a research crew with two agents: a researcher and an analyst.
<div style="text-align:left">
    <img src="images/architecture_runtime.png" width="60%"/>
</div>


### Tutorial Key Features

* Hosting Agents on Amazon Bedrock AgentCore Runtime
* Using Amazon Bedrock models
* Using CrewAI

## Prerequisites

To execute this tutorial you will need:
* Python 3.10+
* uv package manager
* AWS credentials
* Docker running

Further, we need to install a few dependencies: 
* Amazon Bedrock AgentCore SDK
* CrewAI 
* Langchain community package
* Duckduckgo search

We have packaged all necessary dependencies in a pyproject.toml file so they can be installed conveniently. 

In [1]:
# rp-demo changes
# RUN this cell before any other 

import os
PROFILE="sso-genai"
os.environ["AWS_PROFILE"]=PROFILE


MODEL_ID="amazon.nova-pro-v1:0"
# MODEL_ID="amazon.nova-lite-v1:0"
# MODEL_ID="anthropic.claude-haiku-4-5-20251001-v1:0"

# CrewAI (LiteLLM) model identifier used in agents.yaml
CREW_LLM=f"bedrock/{MODEL_ID}"


In [2]:
!uv sync --active --force-reinstall

Resolved 205 packages in 20ms
Prepared 172 packages in 1.05s                                           en-ai-dem⠦ Preparing packages... (171/172)                                               
Uninstalled 30 packages in 1.96s
Installed 172 packages in 1.20s (from file:///Users/rafap/de
 + aiofiles==24.1.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.5
 + aiosignal==1.4.0
 + aiosqlite==0.21.0
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + anyio==4.15.1
 + appdirs==1.4.4
 - appnope==1.0.0
 - asttokens==3.0.2
 + attrs==26.1.0
 + bcrypt==5.0.0
 + beautifulsoup4==4.15.0
 + bedrock-agentcore==1.24.1
 + bedrock-agentcore-starter-toolkit==0.1.1
 + boto3==1.43.114
 + botocore==1.43.114
 + build==1.6.1
 + cel-python==0.5.0
 + certifi==2026.7.22
 + cffi==2.1.1
 + charset-normalizer==3.5.2
 + chromadb==1.5.9
 + click==8.5.0
 - comm==0.2.3
 + crewai==1.15.28
 + crewai-cli==1.15.28
 + crewai-core==1.15.28
 + crewai-tools==1.15.28
 + cryptography==50.0.2
 + ddgs==9.16.0
 - debugpy==1.8.22
 + def

## Creating your multi-agent crew and experimenting locally

Before we deploy our agents to AgentCore Runtime, let's develop and run them locally for experimentation purposes.

In this guide, we’ll walk through creating a research crew that will help us research and analyze a topic, then create a comprehensive report. This practical example demonstrates how AI agents can collaborate to accomplish complex tasks. The example is adapted from a [getting started guide](https://docs.crewai.com/en/guides/crews/first-crew) provided directly by CrewAI.

The local architecture looks as following:

<div style="text-align:left">
    <img src="images/architecture_local.png" width="60%"/>
</div>


### Defining agents, tasks, crew

We will first create the artifacts defining a local CrewAI agent, including: 
* agents.yaml, defining the two agents involved in our crew
* tasks.yaml, defining the tasks to be executed by the agents in our crew
* crew.py, defining our crew consisting of agents working on tasks as defined
* main.py, our local entrypoint kicking off the crew run

In [3]:
import os

os.makedirs("research_crew/config", exist_ok=True)

In [4]:
# rp-demo: original %%writefile cell kept below for reference
# %%writefile research_crew/config/agents.yaml
# researcher:
#   role: >
#     Senior Research Specialist for {topic}
#   goal: >
#     Find comprehensive and accurate information about {topic}
#     with a focus on recent developments and key insights
#   backstory: >
#     You are an experienced research specialist with a talent for
#     finding relevant information from various sources. You excel at
#     organizing information in a clear and structured manner, making
#     complex topics accessible to others.
#   llm: bedrock/global.anthropic.claude-haiku-4-5-20251001-v1:0
#
# analyst:
#   role: >
#     Data Analyst and Report Writer for {topic}
#   goal: >
#     Analyze research findings and create a comprehensive, well-structured
#     report that presents insights in a clear and engaging way
#   backstory: >
#     You are a skilled analyst with a background in data interpretation
#     and technical writing. You have a talent for identifying patterns
#     and extracting meaningful insights from research data, then
#     communicating those insights effectively through well-crafted reports.
#   llm: bedrock/global.anthropic.claude-haiku-4-5-20251001-v1:0

# rp-demo: write agents.yaml from Python so CREW_LLM (from MODEL_ID) is interpolated
import os

agents_yaml = f"""researcher:
  role: >
    Senior Research Specialist for {{topic}}
  goal: >
    Find comprehensive and accurate information about {{topic}}
    with a focus on recent developments and key insights
  backstory: >
    You are an experienced research specialist with a talent for
    finding relevant information from various sources. You excel at
    organizing information in a clear and structured manner, making
    complex topics accessible to others.
  llm: {CREW_LLM}

analyst:
  role: >
    Data Analyst and Report Writer for {{topic}}
  goal: >
    Analyze research findings and create a comprehensive, well-structured
    report that presents insights in a clear and engaging way
  backstory: >
    You are a skilled analyst with a background in data interpretation
    and technical writing. You have a talent for identifying patterns
    and extracting meaningful insights from research data, then
    communicating those insights effectively through well-crafted reports.
  llm: {CREW_LLM}
"""

os.makedirs("research_crew/config", exist_ok=True)
with open("research_crew/config/agents.yaml", "w") as f:
    f.write(agents_yaml)
print(agents_yaml)

researcher:
  role: >
    Senior Research Specialist for {topic}
  goal: >
    Find comprehensive and accurate information about {topic}
    with a focus on recent developments and key insights
  backstory: >
    You are an experienced research specialist with a talent for
    finding relevant information from various sources. You excel at
    organizing information in a clear and structured manner, making
    complex topics accessible to others.
  llm: bedrock/amazon.nova-pro-v1:0

analyst:
  role: >
    Data Analyst and Report Writer for {topic}
  goal: >
    Analyze research findings and create a comprehensive, well-structured
    report that presents insights in a clear and engaging way
  backstory: >
    You are a skilled analyst with a background in data interpretation
    and technical writing. You have a talent for identifying patterns
    and extracting meaningful insights from research data, then
    communicating those insights effectively through well-crafted reports.
  llm

In [5]:
%%writefile research_crew/config/tasks.yaml
research_task:
  description: >
    Conduct thorough research on {topic}. Focus on:
    1. Key concepts and definitions
    2. Historical development and recent trends
    3. Major challenges and opportunities
    4. Notable applications or case studies
    5. Future outlook and potential developments

    Make sure to organize your findings in a structured format with clear sections.
  expected_output: >
    A comprehensive research document with well-organized sections covering
    all the requested aspects of {topic}. Include specific facts, figures,
    and examples where relevant.
  agent: researcher

analysis_task:
  description: >
    Analyze the research findings and create a comprehensive report on {topic}.
    Your report should:
    1. State the topic and begin with an executive summary
    2. Include all key information from the research
    3. Provide insightful analysis of trends and patterns
    4. Offer recommendations or future considerations
    5. Be formatted in a professional, easy-to-read style with clear headings
  expected_output: >
    A polished, professional report on {topic} that presents the research
    findings with added analysis and insights. The report should be well-structured
    with an executive summary, main sections, and conclusion.
  agent: analyst
  context:
    - research_task

Overwriting research_crew/config/tasks.yaml


In [7]:
%%writefile research_crew/crew.py
import logging
import os

from crewai import Agent, Crew, Process, Task
from crewai.project import CrewBase, agent, crew, task
from crewai.agents.agent_builder.base_agent import BaseAgent
from typing import List
from langchain_community.tools import DuckDuckGoSearchRun
from crewai.tools import BaseTool
from crewai_tools import SerperDevTool
from pydantic import Field

# rp-demo: console logging only, opt-in via CREW_LOG=1 (off by default).
# No file logging -> safe for the read-only AgentCore runtime container.
# When enabled, console output is captured by AgentCore and shipped to CloudWatch.
_LOG_FORMAT = "%(asctime)s %(levelname)s %(name)s: %(message)s"
logger = logging.getLogger("research_crew")
if os.environ.get("CREW_LOG") == "1":
    logging.basicConfig(
        level=logging.INFO,
        format=_LOG_FORMAT,
        handlers=[logging.StreamHandler()],
    )
else:
    logger.addHandler(logging.NullHandler())


class SearchTool(BaseTool):
     name: str = "Search"
     description: str = "Useful for searching the web for information."
     search: DuckDuckGoSearchRun = Field(default_factory=DuckDuckGoSearchRun)

     def _run(self, query: str) -> str:
         """Execute the search query and return results"""
         logger.info("SearchTool query: %s", query)
         try:
             return self.search.invoke(query)
         except Exception as e:
             logger.exception("SearchTool failed for query: %s", query)
             return f"Error performing search: {str(e)}"

@CrewBase
class ResearchCrew():
    """Research crew for comprehensive topic analysis and reporting"""

    agents: List[BaseAgent]
    tasks: List[Task]

    @agent
    def researcher(self) -> Agent:
        return Agent(
            config=self.agents_config['researcher'], # type: ignore[index]
            verbose=True,
            tools=[
                #SerperDevTool()
                SearchTool()
                ]
        )

    @agent
    def analyst(self) -> Agent:
        return Agent(
            config=self.agents_config['analyst'], # type: ignore[index]
            verbose=True
        )

    @task
    def research_task(self) -> Task:
        return Task(
            config=self.tasks_config['research_task'] # type: ignore[index]
        )

    @task
    def analysis_task(self) -> Task:
        return Task(
            config=self.tasks_config['analysis_task'], # type: ignore[index]
            #output_file='output/report.md'
        )

    @crew
    def crew(self) -> Crew:
        """Creates the research crew"""
        logger.info("Building research crew (agents=%d, tasks=%d)", len(self.agents), len(self.tasks))
        return Crew(
            agents=self.agents,
            tasks=self.tasks,
            process=Process.sequential,
            verbose=True,
        )

Overwriting research_crew/crew.py


In [8]:
%%writefile research_crew/main.py
import os
from research_crew.crew import ResearchCrew

# Create output directory if it doesn't exist
os.makedirs('output', exist_ok=True)

def run():
    """
    Run the research crew.
    """
    inputs = {
        # 'topic': 'Artificial Intelligence in Healthcare'
        'topic': 'GenAI in Healthcare - impact on patient-doctor interaction'

    }

    # Create and run the crew
    result = ResearchCrew().crew().kickoff(inputs=inputs)

    # Print the result
    print("\n\n=== FINAL REPORT ===\n\n")
    print(result.raw)


if __name__ == "__main__":
    run()

Overwriting research_crew/main.py


### Invoking crew locally

Finally, we can use the CrewAI CLI to lockally kick off the crew. Alternatively, we could also simply run our local entrypoint main.py. This might take a few minutes. 

In [ ]:
!crewai run

## Deploying multi-agent crew to Amazon Bedrock AgentCore

For production-grade agentic applications we will need to run our crew in the cloud. Therefor we will deploy our crew to Amazon Bedrock AgentCore. 

The architecture here will look as following:

<div style="text-align:left">
     <img src="images/architecture_local.png" width="60%"/>
</div>

Deploying the crew to AgentCore takes the following steps: 

### Remote entrypoint

First, we create a remote entrypoint. With AgentCore Runtime, we will decorate the invocation part of our agent with the @app.entrypoint decorator and have it as the entry point for our runtime. This also involves: 
* Import the Runtime App with `from bedrock_agentcore.runtime import BedrockAgentCoreApp`
* Initialize the App in our code with `app = BedrockAgentCoreApp()`
* Decorate the invocation function with the `@app.entrypoint` decorator
* Let AgentCoreRuntime control the running of the agent with `app.run()`

### What happens behind the scenes?

When you use `BedrockAgentCoreApp`, it automatically:

* Creates an HTTP server that listens on the port 8080
* Implements the required `/invocations` endpoint for processing the agent's requirements
* Implements the `/ping` endpoint for health checks (very important for asynchronous agents)
* Handles proper content types and response formats
* Manages error handling according to the AWS standards                                                                                                                                                                        

In [9]:
%%writefile research_crew/research_crew.py
import os
from research_crew.crew import ResearchCrew

# ---------- Agentcore imports --------------------
from bedrock_agentcore.runtime import BedrockAgentCoreApp

app = BedrockAgentCoreApp()
#------------------------------------------------


@app.entrypoint
def agent_invocation(payload, context):
    """Handler for agent invocation"""
    print(f'Payload: {payload}')
    try: 
        # Extract user message from payload with default
        user_message = payload.get("prompt", "Artificial Intelligence in Healthcare")
        print(f"Processing topic: {user_message}")
        
        # Create crew instance and run synchronously
        research_crew_instance = ResearchCrew()
        crew = research_crew_instance.crew()
        
        # Use synchronous kickoff instead of async - this avoids all event loop issues
        result = crew.kickoff(inputs={'topic': user_message})

        print("Context:\n-------\n", context)
        print("Result Raw:\n*******\n", result.raw)
        
        # Safely access json_dict if it exists
        if hasattr(result, 'json_dict'):
            print("Result JSON:\n*******\n", result.json_dict)
        
        return {"result": result.raw}
        
    except Exception as e:
        print(f'Exception occurred: {e}')
        return {"error": f"An error occurred: {str(e)}"}

if __name__ == "__main__":
    app.run()

Overwriting research_crew/research_crew.py


### Deploying the agent to AgentCore Runtime

The `CreateAgentRuntime` operation supports comprehensive configuration options, letting you specify container images, environment variables and encryption settings. You can also configure protocol settings (HTTP, MCP) and authorization mechanisms to control how your clients communicate with the agent. 

**Note:** Operations best practice is to package code as container and push to ECR using CI/CD pipelines and IaC

In this tutorial can will the Amazon Bedrock AgentCode Python SDK to easily package your artifacts and deploy them to AgentCore runtime.

#### Configure AgentCore Runtime deployment

First we will use our starter toolkit to configure the AgentCore Runtime deployment with an entrypoint, the execution role we just created and a requirements file. We will also configure the starter kit to auto create the Amazon ECR repository on launch.

AgentCore configure is required to generate a Dockerfile holding a blueprint for the Docker container the workload will be running in and a .bedrock_agentcore.yaml holding the agentic workload's configuration. During the configure step, your docker file will be generated based on your application code.

<div style="text-align:left">
    <img src="images/configure.png" width="60%"/>
</div>

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()
agent_name = "research_crew_getting_started"
response = agentcore_runtime.configure(
    entrypoint="research_crew/research_crew.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    region=region,
    agent_name=agent_name,
    ## rp-demo change
    platformVersion="V2"
)
response

Entrypoint parsed: file=/Users/rafap/dev/courses/aws-genai/gen-ai-demos/02.agentcore.demos/agentcore-samples/06-workshops/01-AgentCore-runtime/01-hosting-agent/04-crewai-with-bedrock-model/research_crew/research_crew.py, bedrock_agentcore_name=research_crew
Configuring BedrockAgentCore agent: research_crew_getting_started
Generated Dockerfile: /Users/rafap/dev/courses/aws-genai/gen-ai-demos/02.agentcore.demos/agentcore-samples/06-workshops/01-AgentCore-runtime/01-hosting-agent/04-crewai-with-bedrock-model/Dockerfile
Generated .dockerignore: /Users/rafap/dev/courses/aws-genai/gen-ai-demos/02.agentcore.demos/agentcore-samples/06-workshops/01-AgentCore-runtime/01-hosting-agent/04-crewai-with-bedrock-model/.dockerignore
Keeping 'research_crew_getting_started' as default agent
Bedrock AgentCore configured: /Users/rafap/dev/courses/aws-genai/gen-ai-demos/02.agentcore.demos/agentcore-samples/06-workshops/01-AgentCore-runtime/01-hosting-agent/04-crewai-with-bedrock-model/.bedrock_agentcore.yam

ConfigureResult(config_path=PosixPath('/Users/rafap/dev/courses/aws-genai/gen-ai-demos/02.agentcore.demos/agentcore-samples/06-workshops/01-AgentCore-runtime/01-hosting-agent/04-crewai-with-bedrock-model/.bedrock_agentcore.yaml'), dockerfile_path=PosixPath('/Users/rafap/dev/courses/aws-genai/gen-ai-demos/02.agentcore.demos/agentcore-samples/06-workshops/01-AgentCore-runtime/01-hosting-agent/04-crewai-with-bedrock-model/Dockerfile'), dockerignore_path=PosixPath('/Users/rafap/dev/courses/aws-genai/gen-ai-demos/02.agentcore.demos/agentcore-samples/06-workshops/01-AgentCore-runtime/01-hosting-agent/04-crewai-with-bedrock-model/.dockerignore'), runtime='Docker', region='us-east-1', account_id='730558614357', execution_role=None, ecr_repository=None, auto_create_ecr=True)

#### Launching agent to AgentCore Runtime: deploying the remote agentic workload

Now that we've got a docker file, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime. AgentCore launch will then deploy the agentic workload to the cloud. This includes creating a Docker image and pushing it to ECR, as well as getting an endpoint ready for usage.


<div style="text-align:left">
    <img src="images/launch.png" width="85%"/>
</div>

In [14]:
launch_result = agentcore_runtime.launch()

Launching Bedrock AgentCore agent 'research_crew_getting_started' to cloud
Docker image built: bedrock_agentcore-research_crew_getting_started:latest
Using execution role from config: arn:aws:iam::730558614357:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-676668b23d
✅ Execution role validation passed: arn:aws:iam::730558614357:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-676668b23d
Uploading to ECR...
Using ECR repository from config: 730558614357.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-research_crew_getting_started
Authenticating with registry...
Registry authentication successful
Tagging image: bedrock_agentcore-research_crew_getting_started:latest -> 730558614357.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-research_crew_getting_started:latest
Pushing image to registry...


The push refers to repository [730558614357.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-research_crew_getting_started]
d5ff82e92edd: Waiting
4fe813ca9e00: Waiting
4f4fb700ef54: Waiting
387b25ff7b94: Waiting
4dbdca1a513d: Waiting
6e2fcafe72a4: Waiting
3535535e224b: Waiting
85cc15801beb: Waiting
bbeda6b4abb7: Waiting
b944853e0be6: Waiting
6e16323d406e: Waiting
4f4fb700ef54: Waiting
387b25ff7b94: Waiting
4dbdca1a513d: Waiting
6e2fcafe72a4: Waiting
3535535e224b: Waiting
85cc15801beb: Waiting
bbeda6b4abb7: Waiting
b944853e0be6: Waiting
6e16323d406e: Waiting
d5ff82e92edd: Waiting
4fe813ca9e00: Waiting
d5ff82e92edd: Waiting
4fe813ca9e00: Waiting
4f4fb700ef54: Waiting
387b25ff7b94: Waiting
4dbdca1a513d: Waiting
6e2fcafe72a4: Waiting
3535535e224b: Waiting
85cc15801beb: Waiting
bbeda6b4abb7: Waiting
b944853e0be6: Waiting
6e16323d406e: Waiting
bbeda6b4abb7: Waiting
b944853e0be6: Waiting
6e16323d406e: Waiting
d5ff82e92edd: Waiting
4fe813ca9e00: Waiting
4f4fb700ef54: Waiting
387b25ff7b94: Wai

Image pushed successfully
Image uploaded to ECR: 730558614357.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-research_crew_getting_started
Deploying to Bedrock AgentCore...


latest: digest: sha256:fc18b753d082175c4f2791d5cf3af069a2b25da61908b8fbaaca196a122fda45 size: 856


✅ Agent created/updated: arn:aws:bedrock-agentcore:us-east-1:730558614357:runtime/research_crew_getting_started-fBNB7H32kx
Polling for endpoint to be ready...
Agent endpoint: arn:aws:bedrock-agentcore:us-east-1:730558614357:runtime/research_crew_getting_started-fBNB7H32kx/runtime-endpoint/DEFAULT
Deployed to cloud: arn:aws:bedrock-agentcore:us-east-1:730558614357:runtime/research_crew_getting_started-fBNB7H32kx
🔍 Agent logs available at:
   /aws/bedrock-agentcore/runtimes/research_crew_getting_started-fBNB7H32kx-DEFAULT
   /aws/bedrock-agentcore/runtimes/research_crew_getting_started-fBNB7H32kx-DEFAULT/runtime-logs
💡 Tail logs with: aws logs tail /aws/bedrock-agentcore/runtimes/research_crew_getting_started-fBNB7H32kx-DEFAULT --follow
💡 Or view recent logs: aws logs tail /aws/bedrock-agentcore/runtimes/research_crew_getting_started-fBNB7H32kx-DEFAULT --since 1h


#### Checking for the AgentCore Runtime Status

Now that we've deployed the AgentCore Runtime, let's check for it's deployment status

In [ ]:
import time

status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]
end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]
    print(status)
status

Retrieved Bedrock AgentCore status for: research_crew_getting_started


'READY'

### Invoking AgentCore Runtime with boto3

Now that your AgentCore Runtime was created you can invoke it with any AWS SDK. For instance, you can use the boto3 `invoke_agent_runtime` method for it. Since this is a long running agent we are overwriting the default `retries`, `connect_timout`and `read_timeout`.

<div style="text-align:left">
    <img src="images/invoke.png" width=85%"/>
</div>

In [34]:
from botocore.config import Config

# Configure retries and timeout
config = Config(
    retries={
        "max_attempts": 10,  # Increase max retries to 10 (default is 4)
        "mode": "adaptive",  # Options: 'legacy', 'standard', 'adaptive'
    },
    connect_timeout=600,  # Connection timeout in seconds (default is 60)
    read_timeout=3000,  # Read timeout in seconds (default is 60)
)

In [35]:
import boto3
import json
from IPython.display import Markdown, display

agent_arn = launch_result.agent_arn
agentcore_client = boto3.client("bedrock-agentcore", region_name=region)

boto3_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": "What is 2+2?"}),
)

# Capture the runtime session ID for lifecycle management
runtime_session_id = boto3_response.get("runtimeSessionId")
print(f"Runtime Session ID: {runtime_session_id}")

if "text/event-stream" in boto3_response.get("contentType", ""):
    content = []
    for line in boto3_response["response"].iter_lines(chunk_size=1):
        if line:
            line = line.decode("utf-8")
            if line.startswith("data: "):
                line = line[6:]
                print(line)
                content.append(line)
    display(Markdown("\n".join(content)))
else:
    try:
        events = []
        for event in boto3_response.get("response", []):
            events.append(event)
    except Exception as e:
        events = [f"Error reading EventStream: {e}"]
    result = json.loads(events[0].decode("utf-8")) if events else "No response"
    display(Markdown(result if isinstance(result, str) else json.dumps(result, indent=2)))

RuntimeClientError: An error occurred (RuntimeClientError) when calling the InvokeAgentRuntime operation: An error occurred when starting the runtime. Please check your CloudWatch logs for more information.

### Stopping a Session

You'll want to stop individual sessions when they're no longer needed.
This releases the microVM resources for that session while keeping the runtime alive
for new sessions. Below we demonstrate `stop_runtime_session`.

In [ ]:
# --- Inline Session Lifecycle Demo ---
# stop_runtime_session releases the microVM resources for this specific session while keeping the runtime alive for new sessions.

if runtime_session_id:
    agentcore_client.stop_runtime_session(
        agentRuntimeArn=agent_arn,
        runtimeSessionId=runtime_session_id,
        qualifier="DEFAULT",
    )
    print(f"✅ Session '{runtime_session_id}' stopped — microVM resources released")
else:
    print("⚠️ No session ID available to stop")

### Lifecycle Configuration Demo (Active)

Now let's demonstrate how to configure a runtime with a shorter idle timeout.
We'll create a second runtime with a 5-minute (300 second) idle timeout to show
how lifecycle configuration affects session behavior. Both runtimes will coexist.

In [ ]:
# --- Lifecycle Configuration Demo ---
# In production, choose a timeout appropriate for your workload:
#   - Development/testing: 5-15 minutes
#   - Interactive sessions: 30-60 minutes
#   - Long-running workloads: adjust as needed
#

agentcore_runtime_short = Runtime()
agent_name_short = "crewai_claude_short_timeout"

# Configure with shorter idle timeout
response_short = agentcore_runtime_short.configure(
    entrypoint="research_crew/research_crew.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="research_crew/requirements.txt",
    region=region,
    agent_name=agent_name_short,
)

# Launch the second runtime
launch_result_short = agentcore_runtime_short.launch()
print(f"Second runtime launched: {launch_result_short.agent_id}")

# Wait for it to be ready
status_response_short = agentcore_runtime_short.status()
status_short = status_response_short.endpoint["status"]
while status_short not in ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]:
    time.sleep(10)
    status_response_short = agentcore_runtime_short.status()
    status_short = status_response_short.endpoint["status"]
    print(f"Short timeout runtime status: {status_short}")

# Now update the runtime with shorter idle timeout using boto3
# UpdateAgentRuntime is a full-replacement API — we must re-supply all required fields.
# First, retrieve the current runtime configuration.
agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)
current_runtime = agentcore_control_client.get_agent_runtime(agentRuntimeId=launch_result_short.agent_id)

update_response = agentcore_control_client.update_agent_runtime(
    agentRuntimeId=launch_result_short.agent_id,
    agentRuntimeArtifact=current_runtime["agentRuntimeArtifact"],
    roleArn=current_runtime["roleArn"],
    networkConfiguration=current_runtime["networkConfiguration"],
    lifecycleConfiguration={
        "idleRuntimeSessionTimeout": 300  # 5 minutes
    },
)
print("✅ Runtime updated with 5-minute idle timeout")

# Invoke the second runtime to verify it works
invoke_response_short = agentcore_runtime_short.invoke({"prompt": "What is 3+3?"})
print(f"Second runtime response: {invoke_response_short['response'][0]}")

## Cleanup

Let's now clean up the AgentCore Runtime and associated resources. We delete the runtime first to avoid undesired costs, then clean up supporting resources like ECR repositories.

In [ ]:
launch_result.ecr_uri, launch_result.agent_id, launch_result.ecr_uri.split("/")[1]

In [ ]:
# --- Stop active sessions to release microVM resources ---
import boto3

agentcore_client = boto3.client("bedrock-agentcore", region_name=region)
agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)
ecr_client = boto3.client("ecr", region_name=region)

# Stop the active session to release its microVM resources
# In production, this is how you end individual user sessions while keeping the runtime alive
# AgentCore Runtime costs are based on vCPU and Memory — stopping sessions avoids undesired costs
# Note: If the session was already stopped in the earlier demo cell, this will raise a
# ResourceNotFoundException — the except block handles that gracefully.
if "runtime_session_id" in locals() and runtime_session_id:
    try:
        agentcore_client.stop_runtime_session(
            agentRuntimeArn=launch_result.agent_arn,
            runtimeSessionId=runtime_session_id,
            qualifier="DEFAULT",
        )
        print(f"✅ Session '{runtime_session_id}' stopped")
    except Exception as e:
        print(f"⚠️ Failed to stop session '{runtime_session_id}': {e}")

# --- Delete both runtimes ---
# Original runtime
try:
    agentcore_control_client.delete_agent_runtime(
        agentRuntimeId=launch_result.agent_id,
    )
    print(f"✅ Original runtime '{launch_result.agent_id}' deleted")
except Exception as e:
    print(f"⚠️ Failed to delete original runtime: {e}")

# Short-timeout runtime
if "launch_result_short" in locals():
    try:
        agentcore_control_client.delete_agent_runtime(
            agentRuntimeId=launch_result_short.agent_id,
        )
        print(f"✅ Short-timeout runtime '{launch_result_short.agent_id}' deleted")
    except Exception as e:
        print(f"⚠️ Failed to delete short-timeout runtime: {e}")

# --- Delete ECR repositories ---
try:
    ecr_client.delete_repository(repositoryName=launch_result.ecr_uri.split("/")[1], force=True)
    print(f"✅ ECR repository '{launch_result.ecr_uri.split('/')[1]}' deleted")
except Exception as e:
    print(f"⚠️ Failed to delete ECR repository: {e}")

if "launch_result_short" in locals():
    try:
        ecr_client.delete_repository(repositoryName=launch_result_short.ecr_uri.split("/")[1], force=True)
        print(f"✅ Second ECR repository '{launch_result_short.ecr_uri.split('/')[1]}' deleted")
    except Exception as e:
        print(f"⚠️ Failed to delete second ECR repository: {e}")

## Congratulations!